# Business cost modeling and risk decisioning

A calibrated probability is not a business threshold. Actions depend on transaction amount, review effectiveness, and legitimate customer impact. All monetary inputs are **ILLUSTRATIVE / DEMONSTRATION ASSUMPTIONS**, not company facts or actual savings. The frozen 467-feature 80/20 champion and Platt calibrator remain unchanged.


In [1]:
import gc
import json
import sys
import time
from dataclasses import asdict
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from fraud_detection.data import merge_train_data
from fraud_detection.decisioning.costs import expected_action_costs, scenario_profiles
from fraud_detection.decisioning.evaluation import evaluate_policy
from fraud_detection.decisioning.policy import decide
from fraud_detection.features import COMPOSITE_TARGET_ENTITIES
from fraud_detection.models import prepare_engineered_matrices
from fraud_detection.models.calibration import temporal_selection, validate_oof
from fraud_detection.models.ensemble import weighted_score_blend

started = time.perf_counter()
profiles = scenario_profiles()
calibrator = joblib.load(ROOT / "models/probability_calibrator.joblib")
assert calibrator.method == "platt"
print({name: asdict(config) for name, config in profiles.items()})

pd.set_option("display.max_columns", None)

Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\Marwan\\.matplotlib\\fontlist-v3.11.0.json.matplotlib-lock'


{'Balanced': {'fraud_loss_multiplier': 1.0, 'manual_review_cost': 3.0, 'review_fraud_detection_rate': 0.8, 'legitimate_review_friction_cost': 1.0, 'false_decline_cost': 25.0, 'block_operational_cost': 0.5}, 'Fraud-loss-sensitive': {'fraud_loss_multiplier': 2.0, 'manual_review_cost': 3.0, 'review_fraud_detection_rate': 0.8, 'legitimate_review_friction_cost': 1.0, 'false_decline_cost': 20.0, 'block_operational_cost': 0.5}, 'Customer-friction-sensitive': {'fraud_loss_multiplier': 1.0, 'manual_review_cost': 5.0, 'review_fraud_detection_rate': 0.75, 'legitimate_review_friction_cost': 5.0, 'false_decline_cost': 75.0, 'block_operational_cost': 0.5}}


## Fixed expected-cost equations and TRAIN OOF policy analysis

For calibrated probability p and TransactionAmt A: APPROVE = p × A × fraud_loss_multiplier. REVIEW = manual_review_cost + p × (1 − review_fraud_detection_rate) × A × multiplier + (1 − p) × legitimate_review_friction_cost. BLOCK = (1 − p) × false_decline_cost + block_operational_cost. The action with minimum expected cost wins; ties favor APPROVE then REVIEW then BLOCK. Reviewed fraud capture uses the assumed detection rate. The three scenario profiles are fixed in source and are never tuned on labels.


In [2]:
oof = pd.read_csv(ROOT / "reports/ensemble_oof_predictions.csv")
validate_oof(oof)
_, temporal_predictions = temporal_selection(oof)
oof_policy = oof.loc[oof.fold.isin((2, 3)), ["TransactionID", "fold", "y_true"]].reset_index(
    drop=True
)
y_oof, p_oof = temporal_predictions["platt"]
assert np.array_equal(y_oof, oof_policy.y_true.to_numpy())
oof_policy["calibrated_probability"] = p_oof
train_end = int(590540 * 0.70)
train_amounts = pd.read_csv(
    ROOT / "data/raw/train_transaction.csv",
    nrows=train_end,
    usecols=["TransactionID", "TransactionAmt", "isFraud"],
)
oof_policy = oof_policy.merge(train_amounts, on="TransactionID", how="left", validate="one_to_one")
assert oof_policy.TransactionAmt.notna().all()
assert np.array_equal(oof_policy.y_true.to_numpy(), oof_policy.isFraud.to_numpy())
quantiles = train_amounts.TransactionAmt.quantile([0.1, 0.5, 0.9]).to_dict()
print("TRAIN amount quantiles:", quantiles)
print("Leakage-safe policy OOF rows:", len(oof_policy))

C:\fraud-detection-system\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\fraud-detection-system\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


TRAIN amount quantiles: {0.1: 25.95, 0.5: 68.95, 0.9: 280.0}
Leakage-safe policy OOF rows: 82676


In [3]:
train_decisions, train_results = {}, {}
for name, config in profiles.items():
    decided = decide(oof_policy, config)
    train_decisions[name] = decided
    train_results[name] = evaluate_policy(decided, oof_policy.y_true, config)
train_table = pd.DataFrame(
    [{"scenario": name, **metrics} for name, metrics in train_results.items()]
)
display(train_table)
capacity_rates = [("unlimited", None), ("1%", 0.01), ("3%", 0.03), ("5%", 0.05)]
capacity_results = {}
for label, rate in capacity_rates:
    decided = decide(
        oof_policy, profiles["Balanced"], maximum_review_rate=rate, overflow="cheapest"
    )
    capacity_results[label] = evaluate_policy(decided, oof_policy.y_true, profiles["Balanced"])
capacity_table = pd.DataFrame(
    [{"capacity": name, **metrics} for name, metrics in capacity_results.items()]
)
display(capacity_table)

,scenario,n,fraud_count,legitimate_count,fraud_amount_total,fraud_capture_rate,fraud_amount_capture_rate,fraud_transactions_approved,fraud_transactions_reviewed,fraud_transactions_blocked,fraudulent_amount_approved,fraudulent_amount_reviewed,fraudulent_amount_blocked,false_decline_count,false_decline_rate,modeled_total_cost,modeled_cost_per_transaction,approve_all_modeled_cost,approve_all_modeled_cost_per_transaction,modeled_cost_reduction,approve_all_fraud_transactions_missed,approve_all_fraudulent_amount_approved,review_overflow_count,approve_count,approve_rate,legitimate_approve_rate,review_count,review_rate,legitimate_review_rate,block_count,block_rate,legitimate_block_rate
0,Balanced,82676,3358,79318,578094.482,0.696248,0.806320,801,1095,1462,64356.267,238045.626,275692.589,591,0.007451,169003.973452,2.044172,5.268904e+05,6.372955,357886.417745,3358,578094.482,0,67940,0.821762,0.846454,12683,0.153406,0.146095,2053,0.024832,0.007451
1,Fraud-loss-sensitive,82676,3358,79318,578094.482,0.775878,0.874077,562,953,1843,37168.231,178136.822,362789.429,1816,0.022895,246525.084599,2.981822,1.053781e+06,12.745909,807255.697794,3358,578094.482,0,60746,0.734748,0.758769,18271,0.220995,0.218336,3659,0.044257,0.022895
2,Customer-friction-sensitive,82676,3358,79318,578094.482,0.573928,0.683068,1167,1055,1136,121169.323,248188.575,208736.584,199,0.002509,245184.365093,2.965605,5.268904e+05,6.372955,281706.026103,3358,578094.482,0,74964,0.906720,0.930394,6377,0.077132,0.067097,1335,0.016147,0.002509


,capacity,n,fraud_count,legitimate_count,fraud_amount_total,fraud_capture_rate,fraud_amount_capture_rate,fraud_transactions_approved,fraud_transactions_reviewed,fraud_transactions_blocked,fraudulent_amount_approved,fraudulent_amount_reviewed,fraudulent_amount_blocked,false_decline_count,false_decline_rate,modeled_total_cost,modeled_cost_per_transaction,approve_all_modeled_cost,approve_all_modeled_cost_per_transaction,modeled_cost_reduction,approve_all_fraud_transactions_missed,approve_all_fraudulent_amount_approved,review_overflow_count,approve_count,approve_rate,legitimate_approve_rate,review_count,review_rate,legitimate_review_rate,block_count,block_rate,legitimate_block_rate
0,unlimited,82676,3358,79318,578094.482,0.696248,0.806320,801,1095,1462,64356.267,238045.626,275692.589,591,0.007451,169003.973452,2.044172,526890.391196,6.372955,357886.417745,3358,578094.482,0,67940,0.821762,0.846454,12683,0.153406,0.146095,2053,0.024832,0.007451
1,1%,82676,3358,79318,578094.482,0.592079,0.701038,1360,49,1949,166868.450,29798.794,381427.238,2452,0.030914,228245.673710,2.760725,526890.391196,6.372955,298644.717487,3358,578094.482,11857,77449,0.936777,0.959290,826,0.009991,0.009796,4401,0.053232,0.030914
2,3%,82676,3358,79318,578094.482,0.600893,0.713493,1300,201,1857,149408.781,81096.622,347589.079,1645,0.020739,208507.740004,2.521986,526890.391196,6.372955,318382.651192,3358,578094.482,10203,76694,0.927645,0.950528,2480,0.029997,0.028732,3502,0.042358,0.020739
3,5%,82676,3358,79318,578094.482,0.614771,0.734347,1216,388,1754,127211.360,131805.741,319077.381,1197,0.015091,193907.997763,2.345396,526890.391196,6.372955,332982.393433,3358,578094.482,8550,75592,0.914316,0.937694,4133,0.049990,0.047215,2951,0.035694,0.015091


## Review capacity and amount-dependent boundaries

Capacity limits of unlimited, 1%, 3%, and 5% are illustrative sensitivity cases. When REVIEW exceeds capacity, candidates are ranked by expected benefit over the cheaper non-review action, then TransactionID; overflow goes explicitly to that cheaper action. Representative amounts are TRAIN-only 10th, 50th, and 90th percentiles.


In [4]:
p_grid = np.linspace(0, 1, 1001)
boundary_rows = []
for quantile, amount in quantiles.items():
    grid = pd.DataFrame(
        {
            "TransactionID": np.arange(len(p_grid)),
            "calibrated_probability": p_grid,
            "TransactionAmt": amount,
        }
    )
    actions = decide(grid, profiles["Balanced"]).decision.to_numpy()
    starts = np.r_[0, np.flatnonzero(actions[1:] != actions[:-1]) + 1]
    ends = np.r_[starts[1:] - 1, len(actions) - 1]
    for start, end in zip(starts, ends, strict=True):
        boundary_rows.append(
            {
                "amount_quantile": quantile,
                "TransactionAmt": amount,
                "decision": actions[start],
                "p_min": p_grid[start],
                "p_max": p_grid[end],
            }
        )
boundaries = pd.DataFrame(boundary_rows)
display(boundaries)
amount_grid = np.geomspace(
    max(0.01, float(train_amounts.TransactionAmt.quantile(0.01))),
    max(1.0, float(train_amounts.TransactionAmt.quantile(0.99))),
    120,
)
p_region = np.linspace(0, 1, 200)
p_mesh, amount_mesh = np.meshgrid(p_region, amount_grid)
a, r, b = expected_action_costs(p_mesh.ravel(), amount_mesh.ravel(), profiles["Balanced"])
regions = np.argmin(np.column_stack((a, r, b)), axis=1).reshape(p_mesh.shape)
fig, ax = plt.subplots(figsize=(8, 5))
ax.pcolormesh(
    p_region,
    amount_grid,
    regions,
    shading="nearest",
    cmap=ListedColormap(["#4c9f70", "#e5ac45", "#c86262"]),
    vmin=0,
    vmax=2,
)
ax.set_yscale("log")
ax.set(
    xlabel="Calibrated fraud probability",
    ylabel="TransactionAmt (log scale)",
    title="Balanced illustrative minimum-cost decision regions",
)
ax.legend(
    handles=[
        Patch(color=color, label=name)
        for color, name in [("#4c9f70", "APPROVE"), ("#e5ac45", "REVIEW"), ("#c86262", "BLOCK")]
    ]
)
fig.tight_layout()
fig.savefig(ROOT / "reports/figures/probability_vs_amount_decision_regions.png", dpi=160)
plt.close(fig)

,amount_quantile,TransactionAmt,decision,p_min,p_max
0,0.1,25.95,APPROVE,0.000,0.183
1,0.1,25.95,REVIEW,0.184,0.736
2,0.1,25.95,BLOCK,0.737,1.000
3,0.5,68.95,APPROVE,0.000,0.071
4,0.5,68.95,REVIEW,0.072,0.568
5,0.5,68.95,BLOCK,0.569,1.000
6,0.9,280.00,APPROVE,0.000,0.017
7,0.9,280.00,REVIEW,0.018,0.268
8,0.9,280.00,BLOCK,0.269,1.000


In [5]:
names = list(profiles)
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(names))
bottom = np.zeros(len(names))
for action, color in [("approve", "#4c9f70"), ("review", "#e5ac45"), ("block", "#c86262")]:
    rates = np.array([train_results[name][f"{action}_rate"] for name in names])
    ax.bar(x, rates, bottom=bottom, color=color, label=action.upper())
    bottom += rates
ax.set_xticks(x, names, rotation=10)
ax.set(ylabel="Share of TRAIN OOF transactions", ylim=(0, 1), title="Decision workload by scenario")
ax.legend()
fig.tight_layout()
fig.savefig(ROOT / "reports/figures/business_scenario_decision_rates.png", dpi=160)
plt.close(fig)
fig, ax = plt.subplots(figsize=(7, 5))
for name in names:
    m = train_results[name]
    impact = m["legitimate_review_rate"] + m["legitimate_block_rate"]
    ax.scatter(impact, m["fraud_capture_rate"], s=80)
    ax.annotate(name, (impact, m["fraud_capture_rate"]), xytext=(5, 5), textcoords="offset points")
ax.set(
    xlabel="Legitimate review + block rate",
    ylabel="Modeled fraud transaction capture",
    title="Fraud capture versus customer impact (TRAIN OOF)",
)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(ROOT / "reports/figures/fraud_capture_vs_customer_impact.png", dpi=160)
plt.close(fig)
print("Policy, profiles, and capacities frozen before external VALIDATION")

Policy, profiles, and capacities frozen before external VALIDATION


In [6]:
oof_policy_rows = len(oof_policy)
del train_decisions, oof_policy, oof, temporal_predictions, train_amounts, decided, grid
del p_mesh, amount_mesh, regions, a, r, b
gc.collect()
print("Released TRAIN policy arrays before champion rebuild")

Released TRAIN policy arrays before champion rebuild


## External VALIDATION, once after policy freeze

The earlier notebook did not persist validation scores. Rebuild its exact full-TRAIN 467-feature champion with original numeric types, frozen parameters, and 80/20 weights; then apply the existing serialized Platt calibrator. No policy assumptions are changed afterward. HOLDOUT is excluded.


In [7]:
full_rows = 590540
train_end = int(full_rows * 0.70)
validation_end = train_end + int(full_rows * 0.15)
transaction_path = ROOT / "data/raw/train_transaction.csv"
sample = pd.read_csv(transaction_path, nrows=1000, low_memory=False)
dtypes = {
    name: ("float64" if pd.api.types.is_numeric_dtype(dtype) else "object")
    for name, dtype in sample.dtypes.items()
}
dtypes["TransactionID"] = "int64"
dtypes["TransactionDT"] = "int64"
dtypes["card1"] = "int64"
dtypes["isFraud"] = "int8"
del sample
transaction = pd.read_csv(transaction_path, nrows=validation_end, dtype=dtypes)
validation_max_id = int(transaction.TransactionID.iloc[-1])
identity_path = ROOT / "data/raw/train_identity.csv"
identity_rows_through_validation = 125565  # Fixed IEEE-CIS chronological split boundary.
identity = pd.read_csv(identity_path, nrows=identity_rows_through_validation, low_memory=False)
assert int(identity.TransactionID.iloc[-1]) <= validation_max_id
merged = merge_train_data(transaction, identity)
train = merged.iloc[:train_end]
validation = merged.iloc[train_end:validation_end]
del transaction, identity, merged
gc.collect()
prepared = prepare_engineered_matrices(
    train,
    validation,
    include_graph=False,
    include_composite=True,
    include_target_encoding=True,
    scaling_batch_rows=len(train),
)
positions = {name: i for i, name in enumerate(prepared.feature_names)}
baseline_names = tuple(
    name for name in prepared.feature_names if not name.endswith("_target_encoded_fraud_rate")
)
composite_te_features = tuple(
    f"{entity}_target_encoded_fraud_rate" for entity in COMPOSITE_TARGET_ENTITIES
)
selected_names = baseline_names + composite_te_features
assert len(selected_names) == 467 and len(set(selected_names)) == 467
assert "isFraud" not in selected_names and "y_true" not in selected_names
indices = np.asarray([positions[name] for name in selected_names], dtype=np.int32)
x_train, x_validation = prepared.x_train[:, indices], prepared.x_evaluation[:, indices]
y_train = train.isFraud.astype("int8")
y_validation = validation.isFraud.astype("int8")
matrix_mib = (prepared.x_train.nbytes + prepared.x_evaluation.nbytes) / 1024**2
lgbm = lgb.LGBMClassifier(
    learning_rate=0.1,
    n_estimators=400,
    num_leaves=63,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)
lgbm.fit(x_train, y_train)
lgbm_score = lgbm.predict_proba(x_validation)[:, 1]
scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
xgboost = xgb.XGBClassifier(
    learning_rate=0.1,
    max_depth=7,
    n_estimators=300,
    tree_method="hist",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
)
xgboost.fit(x_train, y_train)
xgboost_score = xgboost.predict_proba(x_validation)[:, 1]
raw_validation = weighted_score_blend(
    lgbm_score, xgboost_score, lightgbm_weight=0.8, xgboost_weight=0.2
)
calibrated_validation = calibrator.transform(raw_validation)
assert abs(average_precision_score(y_validation, raw_validation) - 0.615265) < 0.000002
assert abs(roc_auc_score(y_validation, raw_validation) - 0.916224) < 0.000002

assert len(selected_names) == 467

In [8]:
validation_decisions, validation_results = {}, {}
for name, config in profiles.items():
    frame = pd.DataFrame(
        {
            "TransactionID": validation.TransactionID.to_numpy(),
            "calibrated_probability": calibrated_validation,
            "TransactionAmt": validation.TransactionAmt.to_numpy(),
        }
    )
    decided = decide(frame, config)
    validation_decisions[name] = decided
    validation_results[name] = evaluate_policy(decided, y_validation, config)
validation_table = pd.DataFrame(
    [{"scenario": name, **metrics} for name, metrics in validation_results.items()]
)
display(validation_table)

,scenario,n,fraud_count,legitimate_count,fraud_amount_total,fraud_capture_rate,fraud_amount_capture_rate,fraud_transactions_approved,fraud_transactions_reviewed,fraud_transactions_blocked,fraudulent_amount_approved,fraudulent_amount_reviewed,fraudulent_amount_blocked,false_decline_count,false_decline_rate,modeled_total_cost,modeled_cost_per_transaction,approve_all_modeled_cost,approve_all_modeled_cost_per_transaction,modeled_cost_reduction,approve_all_fraud_transactions_missed,approve_all_fraudulent_amount_approved,review_overflow_count,approve_count,approve_rate,legitimate_approve_rate,review_count,review_rate,legitimate_review_rate,block_count,block_rate,legitimate_block_rate
0,Balanced,88581,3042,85539,496907.588,0.690138,0.817614,709,1168,1165,49739.884,204445.950,242721.754,475,0.005553,179424.542932,2.025542,5.067947e+05,5.721257,327370.111238,3042,496907.588,0,73908,0.834355,0.855738,13033,0.147131,0.138709,1640,0.018514,0.005553
1,Fraud-loss-sensitive,88581,3042,85539,496907.588,0.767916,0.877620,509,985,1548,32136.505,143375.354,321395.729,1772,0.020716,264301.841757,2.983731,1.013589e+06,11.442514,749287.466583,3042,496907.588,0,65946,0.744471,0.764996,19315,0.218049,0.214288,3320,0.037480,0.020716
2,Customer-friction-sensitive,88581,3042,85539,496907.588,0.543475,0.686116,1114,1099,829,100249.073,222889.907,173768.608,117,0.001368,256077.440371,2.890885,5.067947e+05,5.721257,250717.213799,3042,496907.588,0,81173,0.916370,0.935936,6462,0.072950,0.062697,946,0.010679,0.001368


In [9]:
tracking_uri = f"sqlite:///{(ROOT / 'model_comparison.db').resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("fraud-detection-business-decisioning")
tags = {
    "stage": "business-decisioning",
    "dataset": "ieee-cis",
    "model": "calibrated-ensemble",
    "business_values": "illustrative",
}
run_ids = {}
key_metrics = (
    "approve_rate",
    "review_rate",
    "block_rate",
    "fraud_capture_rate",
    "fraud_amount_capture_rate",
    "legitimate_review_rate",
    "legitimate_block_rate",
    "modeled_total_cost",
    "modeled_cost_per_transaction",
)
for name, config in profiles.items():
    with mlflow.start_run(run_name=f"scenario-{name}") as run:
        mlflow.set_tags(tags)
        mlflow.log_params({"scenario_name": name, **asdict(config)})
        mlflow.log_metrics({f"train_{key}": float(train_results[name][key]) for key in key_metrics})
        mlflow.log_metrics(
            {f"validation_{key}": float(validation_results[name][key]) for key in key_metrics}
        )
        run_ids[name] = run.info.run_id
for label, rate in capacity_rates:
    with mlflow.start_run(run_name=f"balanced-capacity-{label}") as run:
        mlflow.set_tags(tags)
        mlflow.log_params(
            {
                "scenario_name": "Balanced",
                "maximum_review_rate": str(rate),
                "overflow": "cheapest",
                **asdict(profiles["Balanced"]),
            }
        )
        mlflow.log_metrics(
            {f"train_{key}": float(capacity_results[label][key]) for key in key_metrics}
        )
        run_ids[f"capacity-{label}"] = run.info.run_id
print(run_ids)

{'Balanced': 'c5a1857447de486ba0cf4c8d5067d16f', 'Fraud-loss-sensitive': 'a18fd95b2c9b4d7980da7d36987d0fea', 'Customer-friction-sensitive': 'ef32f3c78caf4e5c87efe4a43289b120', 'capacity-unlimited': '459a16f80c634a249fb82ad97b10865f', 'capacity-1%': 'e7f7c66d5c294d46948c5f56c54d29b1', 'capacity-3%': 'e5d20f555af84f2b8ae714402c87f7fb', 'capacity-5%': '0a538ff8be084dd484f88688fd5195e7'}


In [10]:
def markdown_table(frame, columns):
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for row in frame.loc[:, columns].itertuples(index=False, name=None):
        lines.append(
            "| " + " | ".join(f"{v:.4f}" if isinstance(v, float) else str(v) for v in row) + " |"
        )
    return "\n".join(lines)


scenario_columns = [
    "scenario",
    "approve_rate",
    "review_rate",
    "block_rate",
    "fraud_capture_rate",
    "fraud_amount_capture_rate",
    "legitimate_review_rate",
    "legitimate_block_rate",
    "false_decline_count",
    "approve_all_modeled_cost",
    "modeled_total_cost",
    "modeled_cost_reduction",
]
detail_columns = [
    "scenario",
    "approve_count",
    "review_count",
    "block_count",
    "fraud_transactions_approved",
    "fraud_transactions_reviewed",
    "fraud_transactions_blocked",
    "fraudulent_amount_approved",
    "fraudulent_amount_reviewed",
    "fraudulent_amount_blocked",
    "legitimate_approve_rate",
    "legitimate_review_rate",
    "legitimate_block_rate",
    "modeled_cost_per_transaction",
    "approve_all_modeled_cost_per_transaction",
]
capacity_columns = [
    "capacity",
    "review_rate",
    "fraud_capture_rate",
    "fraud_amount_capture_rate",
    "false_decline_count",
    "modeled_total_cost",
]
profile_lines = "\n".join(f"- **{name}:** {asdict(config)}" for name, config in profiles.items())
report = f"""# Business cost modeling and risk decisioning

## MODEL/DATA MEASUREMENTS

The frozen 467-feature champion is 80% LightGBM and 20% XGBoost, followed by the previously selected Platt calibrator. TRAIN OOF policy analysis uses only Fold 2 calibrated from Fold 1 and Fold 3 calibrated from Folds 1+2, giving {oof_policy_rows:,} leakage-safe rows. TransactionAmt and labels were aligned by unique TransactionID, and OOF labels were checked against TRAIN labels. External VALIDATION was accessed only after all policies and scenarios were frozen. HOLDOUT was not loaded or evaluated.

## ILLUSTRATIVE BUSINESS COST ASSUMPTIONS

These values are **ILLUSTRATIVE / DEMONSTRATION ASSUMPTIONS**, not company values. IEEE-CIS does not supply actual review cost, recovery, false-decline cost, or customer-friction cost. Replace these inputs with measured, approved values before operational use. No scenario was tuned on TRAIN or external VALIDATION outcomes.

{profile_lines}

The Balanced profile uses a unit fraud loss multiplier and moderate review and false-decline costs as a neutral demonstration. Fraud-loss-sensitive doubles the modeled fraud exposure and lowers false-decline cost, which favors intervention. Customer-friction-sensitive raises review friction and false-decline cost, which favors approval. The review detection rates are illustrative effectiveness assumptions, not measurements.

## Expected-cost equations and action rule

For p = calibrated fraud probability, A = TransactionAmt, m = fraud loss multiplier, d = review detection rate:

- APPROVE = p × A × m. Fraud exposure is probability times amount times loss multiplier.
- REVIEW = manual review cost + p × (1 − d) × A × m + (1 − p) × legitimate review friction cost. Residual fraud loss remains when review fails.
- BLOCK = (1 − p) × false decline cost + block operational cost. Legitimate customer loss is modeled probabilistically.

Choose minimum expected cost. Ties follow APPROVE, REVIEW, BLOCK. Optional review capacity ranks candidates by expected benefit over the cheaper non-review action; ties use TransactionID. Overflow goes explicitly to the cheaper of APPROVE or BLOCK. No fixed probability threshold is used.

## TRAIN OOF policy analysis

{markdown_table(train_table, scenario_columns)}

Detailed counts, fraud amounts, legitimate impact, and per-transaction costs:

{markdown_table(train_table, detail_columns)}

Fraud capture includes blocked fraud plus reviewed fraud times the illustrative review detection rate. Fraud amount capture applies the same rule to fraudulent TransactionAmt. These are modeled outcomes, not observed reviewer performance. Detailed fraud approved/reviewed/blocked counts and amounts and legitimate rates are in the executed notebook. Approve-all misses all {train_results["Balanced"]["fraud_count"]:,} fraud transactions and approves {train_results["Balanced"]["fraud_amount_total"]:.2f} fraudulent amount. The approve-all modeled reference varies by scenario multiplier. Cost reductions are **ILLUSTRATIVE SCENARIO OUTPUTS**, not actual savings.

## Balanced review capacity sensitivity (TRAIN OOF)

{markdown_table(capacity_table, capacity_columns)}

Capacities are sensitivity settings, not optimized values. Review overflow counts are in the notebook.

## Representative amount decision boundaries

Low, median, high amounts are TRAIN TransactionAmt quantiles 10%, 50%, 90%: {quantiles}. Regions are approximate to a 0.001 probability grid for Balanced assumptions.

{markdown_table(boundaries, ["amount_quantile", "TransactionAmt", "decision", "p_min", "p_max"])}

See the three decisioning figures in reports/figures/.

## External VALIDATION: one assessment of frozen scenarios

{markdown_table(validation_table, scenario_columns)}

Detailed external VALIDATION outcomes:

{markdown_table(validation_table, detail_columns)}

Approve-all on external VALIDATION misses all {validation_results["Balanced"]["fraud_count"]:,} fraud transactions and approves {validation_results["Balanced"]["fraud_amount_total"]:.2f} fraudulent amount. Decisions use only Platt-calibrated probabilities. Detailed validation fraud counts and fraudulent amount totals, and legitimate approval/review/block rates, are in the notebook. The capture trade-offs depend on illustrative costs and assumed review effectiveness; they are not measured business savings.

## Operational limits and production implications

Real review detection, recovery, chargeback, customer lifetime value, and friction costs are unknown. Expected cost is a scenario calculation using calibrated probabilities, not realized net revenue. Review capacity changes the action mix and can increase modeled cost. Controlled use requires approved cost inputs and matured outcome labels. No API, extra rules, SHAP, or holdout evaluation was performed.

## Runtime, memory, tracking

Notebook runtime: {time.perf_counter() - started:.1f} seconds. Prepared feature matrices: {matrix_mib:.1f} MiB; the notebook rebuild required several GiB of process memory, so TRAIN OOF arrays were released before model preparation. MLflow experiment: fraud-detection-business-decisioning; tracking URI: {tracking_uri}; runs: {json.dumps(run_ids)}.
"""
(ROOT / "reports/business_decisioning_summary.md").write_text(report, encoding="utf-8")
print("Report written; runtime seconds:", time.perf_counter() - started)

Report written; runtime seconds: 197.275615999999
